# Sensor Repair — Notebook 02: Training and validation pilot

**All executable code is in one cell. Use a GPU runtime; a T4 is sufficient for the model sizes.**

1. Keep Notebook 01 outputs in `My Drive/SensorRepair_ICDSIS/processed/v1`.
2. Select a GPU runtime, run the single code cell, and authorize Drive access.
3. If interrupted, rerun the same cell with the same settings and software environment. Completed epochs and cached conditions are reused.
4. Share the final output beginning `NOTEBOOK 02 COMPLETE` and these files from `My Drive/SensorRepair_ICDSIS/outputs/notebook02_v1`:
   - `notebook02_pilot_summary.csv`
   - `notebook02_run_report.json`
   - `notebook02_validation_comparison.csv`

The cell trains a small activity classifier and sensor reconstructor on **both datasets**, using **fold_0 and seed 2026**. It compares six deployable baseline rules and one label-assisted oracle diagnostic across clean data and nine development-fault conditions. It also saves action predictions from the policy-training partition for Notebook 03.

The learned keep/mask/reconstruct policy will be trained in Notebook 03. This is a validation pilot; the final study requires all four DSADS participant folds and repeated model seeds. Final test predictions and held-out stuck-fault evaluation remain reserved until the procedure is frozen.

Masking is an intentional action with an availability indicator. Injected fault locations are hidden from the predictor and future policy features. The two datasets are trained independently. Every reported pilot F1 uses a 0–1 scale in CSV files; the final printed table uses percentages.

Models are selected using validation data. Reconstruction thresholds use the 95th percentile of residuals on clean policy-partition signals. The oracle uses labels for diagnosis and is not a deployable baseline. Candidate-generation timings include all action evaluations and should not be interpreted as per-baseline latency.

Implementation checks used real-data subsets and short CPU training, including checkpoint interruption/resumption and cache reuse. Your full GPU run supplies the actual pilot results.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import hashlib
import json
import math
import random
import shutil
import sys
import time
import zlib
import gc
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

PROJECT = Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
PREPARED = PROJECT / 'processed' / 'v1'
OUTPUT = PROJECT / 'outputs' / 'notebook02_v1'
LOCAL = Path('/content/sensorrepair_notebook02_v1')
REQUIRE_GPU = True
SEED = 2026
FOLD = 'fold_0'
CLASSIFIER_EPOCHS = 30
RECONSTRUCTOR_EPOCHS = 25
PATIENCE = 6
BATCH_SIZE = 128
INFERENCE_BATCH_SIZE = 128
LEARNING_RATE = 0.001
CODE_VERSION = 'sensorrepair_nb02_v1'
CELL_SIGNATURE = 'bcd28764d2f9dc0381fe87cfe97a517352ec74eff179ed11d3da97bff8ba7437'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and DEVICE.type != 'cuda':
    raise RuntimeError('Select a GPU runtime (T4 is sufficient), reconnect, and rerun this same cell.')
OUTPUT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False
print('NOTEBOOK 02 | two-dataset validation pilot | one executable cell', flush=True)
print('Device:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU', '| PyTorch:', torch.__version__)
print('Scope: fold_0, seed 2026; development faults only; final test predictions reserved.')


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    partial = path.with_name(path.name + '.partial')
    partial.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(partial, path)


def save_csv(path, frame):
    path = Path(path)
    partial = path.with_name(path.name + '.partial')
    frame.to_csv(partial, index=False)
    os.replace(partial, path)


def save_npz(path, **arrays):
    path = Path(path)
    partial = path.with_name(path.name + '.partial')
    with partial.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(partial, path)


def save_checkpoint(path, state):
    path = Path(path)
    temporary = LOCAL / (path.name + '.tmp')
    torch.save(state, temporary)
    partial = path.with_name(path.name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def load_checkpoint(path):
    return torch.load(path, map_location='cpu', weights_only=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)


def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


required_reports = ['notebook01_complete.json', 'protocol.json', 'data_integrity.json', 'artifact_inventory.json']
for filename in required_reports:
    if not (PREPARED / filename).is_file():
        raise FileNotFoundError(f'Missing {filename}. Complete Notebook 01 in {PREPARED}.')
completion = json.loads((PREPARED / 'notebook01_complete.json').read_text())
protocol = json.loads((PREPARED / 'protocol.json').read_text())
integrity = json.loads((PREPARED / 'data_integrity.json').read_text())
assert completion['ready_for_training'] and all(integrity[n]['ready_for_training'] for n in ['dsads', 'har'])
assert sha256_file(PREPARED / 'protocol.json') == completion['protocol_sha256'], 'Protocol changed after preparation.'
assert sha256_file(PREPARED / 'artifact_inventory.json') == completion['artifact_inventory_sha256']
assert protocol['development_fault_types'] == ['dropout', 'bias', 'gain']
assert protocol['held_out_fault_type'] == 'stuck'
inventory = {record['filename']: record for record in json.loads((PREPARED / 'artifact_inventory.json').read_text())}


def cached_input(filename):
    expected = inventory[filename]['sha256']
    target = LOCAL / 'inputs' / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256_file(target) == expected:
        return target
    source = PREPARED / filename
    if not source.exists():
        raise FileNotFoundError(str(source))
    temporary = target.with_name(target.name + '.partial')
    shutil.copyfile(source, temporary)
    assert sha256_file(temporary) == expected, f'Input hash mismatch: {filename}'
    os.replace(temporary, target)
    return target


class ActivityCNN(nn.Module):
    def __init__(self, channels, classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 7, padding=3), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.GroupNorm(8, 96), nn.GELU(),
            nn.Conv1d(96, 128, 5, stride=2, padding=2), nn.GroupNorm(8, 128), nn.GELU(),
            nn.AvgPool1d(kernel_size=8, stride=8), nn.Flatten(),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, classes))

    def forward(self, values, available=None):
        if available is None:
            available = torch.ones_like(values)
        return self.encoder(torch.cat([values, available], dim=1))


class SensorReconstructor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 5, padding=2), nn.GELU(),
            nn.Conv1d(64, 96, 5, padding=2), nn.GELU(),
            nn.Conv1d(96, 64, 5, padding=2), nn.GELU(), nn.Conv1d(64, channels, 1))

    def forward(self, values, available):
        return self.network(torch.cat([values, available], dim=1))


def corrupt_signals(values, groups, rng, fault=None, severity=None):
    # Fault locations and severity metadata are never returned as model features.
    result = np.array(values, dtype=np.float32, copy=True)
    for i in range(len(result)):
        kind = fault if fault is not None else str(rng.choice(['dropout', 'bias', 'gain']))
        level = int(severity) if severity is not None else int(rng.integers(0, 3))
        group = groups[int(rng.integers(len(groups)))]['channels']
        if kind == 'dropout':
            fraction = protocol['fault_severities']['dropout']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            result[i, group, start:start + length] = 0.0
        elif kind == 'bias':
            amount = protocol['fault_severities']['bias']['absolute_offset_in_training_std'][level]
            signs = rng.choice([-1.0, 1.0], size=len(group)).astype(np.float32)
            result[i, group, :] += np.float32(amount) * signs[:, None]
        elif kind == 'gain':
            factor = float(protocol['fault_severities']['gain']['factor'][level])
            factor = factor if rng.random() < 0.5 else 1.0 / factor
            result[i, group, :] *= np.float32(factor)
        else:
            raise ValueError('Notebook 02 only uses dropout, bias and gain faults.')
    return result


def mask_groups(values, groups, rng):
    masked = np.array(values, dtype=np.float32, copy=True)
    available = np.ones_like(masked)
    for i in range(len(masked)):
        channels = groups[int(rng.integers(len(groups)))]['channels']
        masked[i, channels, :] = 0.0
        available[i, channels, :] = 0.0
    return masked, available


def classifier_epoch_inputs(clean, groups, seed):
    rng = np.random.default_rng(seed)
    values = np.array(clean, copy=True)
    available = np.ones_like(values)
    choices = rng.random(len(values))
    faults = np.flatnonzero((choices >= 0.35) & (choices < 0.80))
    masks = np.flatnonzero(choices >= 0.80)
    values[faults] = corrupt_signals(clean[faults], groups, rng)
    masked, indicators = mask_groups(clean[masks], groups, rng)
    values[masks], available[masks] = masked, indicators
    return values, available


@torch.inference_mode()
def classifier_probabilities(model, values, available=None):
    model.eval()
    outputs = []
    for start in range(0, len(values), INFERENCE_BATCH_SIZE):
        xb = torch.as_tensor(values[start:start + INFERENCE_BATCH_SIZE], device=DEVICE)
        mb = None if available is None else torch.as_tensor(available[start:start + INFERENCE_BATCH_SIZE], device=DEVICE)
        outputs.append(model(xb, mb).softmax(dim=1).cpu().numpy())
    return np.concatenate(outputs)


def macro_f1(y, prediction, classes):
    return float(f1_score(y, prediction, average='macro', labels=np.arange(classes), zero_division=0))


@torch.inference_mode()
def reconstruction_validation(model, values, groups):
    model.eval()
    total, count = 0.0, 0
    for start in range(0, len(values), INFERENCE_BATCH_SIZE):
        clean = torch.as_tensor(values[start:start + INFERENCE_BATCH_SIZE], device=DEVICE)
        for group in groups:
            channels = group['channels']
            observed, available = clean.clone(), torch.ones_like(clean)
            observed[:, channels, :], available[:, channels, :] = 0.0, 0.0
            prediction = model(observed, available)
            errors = (prediction[:, channels, :] - clean[:, channels, :]).square()
            total += float(errors.sum().item())
            count += errors.numel()
    return total / count


def cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def fit_model(kind, model, clean_train, train_y, validation, val_y, groups, classes, run, signature, stage_seed):
    seed_everything(stage_seed)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)
    last_file, best_file = run / f'{kind}_last.pt', run / f'{kind}_best.pt'
    maximum = CLASSIFIER_EPOCHS if kind == 'classifier' else RECONSTRUCTOR_EPOCHS
    history, first_epoch, stale, best_score = [], 1, 0, -1e30
    if last_file.exists():
        checkpoint = load_checkpoint(last_file)
        assert checkpoint['signature'] == signature, 'Checkpoint configuration changed; use a new output version.'
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        history, first_epoch = checkpoint['history'], checkpoint['epoch'] + 1
        stale, best_score = checkpoint['stale'], checkpoint['best_score']
        torch.set_rng_state(checkpoint['torch_rng'])
        if DEVICE.type == 'cuda' and checkpoint['cuda_rng']:
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        if checkpoint['complete']:
            best = load_checkpoint(best_file)
            assert best['signature'] == signature
            model.load_state_dict(best['model'])
            print(f'  {kind}: reused completed checkpoint (best epoch {best["epoch"]}).', flush=True)
            return model, history
        print(f'  {kind}: resuming at epoch {first_epoch}.', flush=True)
    counts = np.bincount(train_y, minlength=classes)
    weights = 1.0 / np.sqrt(np.maximum(counts, 1))
    weights = torch.as_tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)
    validation_cases = [validation]
    if kind == 'classifier':
        for number, fault in enumerate(['dropout', 'bias', 'gain']):
            validation_cases.append(corrupt_signals(validation, groups, np.random.default_rng(stage_seed + 800 + number), fault, 1))
    reconstruction_indices = np.random.default_rng(stage_seed + 500).permutation(len(validation))[:256]
    elapsed_before = history[-1]['cumulative_seconds'] if history else 0.0
    started = time.perf_counter()
    for epoch in range(first_epoch, maximum + 1):
        epoch_started = time.perf_counter()
        lr = LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * (epoch - 1) / maximum)))
        for group in optimizer.param_groups:
            group['lr'] = lr
        if kind == 'classifier':
            values, available = classifier_epoch_inputs(clean_train, groups, stage_seed + epoch * 1000)
            target = torch.from_numpy(train_y)
        else:
            values, available = mask_groups(clean_train, groups, np.random.default_rng(stage_seed + epoch * 1000))
            target = torch.from_numpy(clean_train)
        generator = torch.Generator().manual_seed(stage_seed + epoch)
        loader = DataLoader(TensorDataset(torch.from_numpy(values), torch.from_numpy(available), target),
                            batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                            num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
        model.train()
        loss_sum, seen = 0.0, 0
        for xb, mb, yb in loader:
            xb, mb, yb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction = model(xb, mb)
            if kind == 'classifier':
                log_probability = prediction.log_softmax(dim=1).gather(1, yb[:, None]).squeeze(1)
                example_weight = weights[yb]
                loss = -(log_probability * example_weight).sum() / example_weight.sum()
            else:
                missing = 1.0 - mb
                loss = ((prediction - yb).square() * missing).sum() / missing.sum().clamp_min(1.0)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite {kind} loss; the last completed epoch is saved.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += float(loss.item()) * len(xb)
            seen += len(xb)
        if kind == 'classifier':
            scores = [macro_f1(val_y, classifier_probabilities(model, case).argmax(axis=1), classes)
                      for case in validation_cases]
            score, val_measure = float(np.mean(scores)), float(np.mean(scores))
        else:
            val_measure = reconstruction_validation(model, validation[reconstruction_indices], groups)
            score = -val_measure
        improved = score > best_score + 1e-6
        if improved:
            best_score, stale = score, 0
            save_checkpoint(best_file, {'signature': signature, 'epoch': epoch, 'model': cpu_state(model),
                                        'validation_measure': val_measure})
        else:
            stale += 1
        cuda_sync()
        history.append({'epoch': epoch, 'train_loss': loss_sum / seen, 'validation_measure': val_measure,
                        'epoch_seconds': time.perf_counter() - epoch_started,
                        'cumulative_seconds': elapsed_before + time.perf_counter() - started, 'learning_rate': lr})
        complete = epoch == maximum or stale >= PATIENCE
        save_checkpoint(last_file, {'signature': signature, 'epoch': epoch, 'model': cpu_state(model),
                                    'optimizer': optimizer.state_dict(), 'best_score': best_score,
                                    'stale': stale, 'history': history, 'complete': complete,
                                    'torch_rng': torch.get_rng_state(),
                                    'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else []})
        save_csv(run / f'{kind}_history.csv', pd.DataFrame(history))
        if epoch == 1 or epoch % 5 == 0 or complete:
            label = 'validation mean Macro-F1' if kind == 'classifier' else 'validation masked MSE'
            print(f'  {kind} epoch {epoch:02d}/{maximum} | loss {loss_sum / seen:.4f} | {label} {val_measure:.4f}', flush=True)
        del values, available, target, loader
        if complete:
            break
    best = load_checkpoint(best_file)
    assert best['signature'] == signature
    model.load_state_dict(best['model'])
    return model, history


@torch.inference_mode()
def action_outputs(classifier, reconstructor, values, groups, classes):
    classifier.eval()
    reconstructor.eval()
    number = len(values)
    group_count = len(groups)
    probabilities = np.empty((number, 1 + 2 * group_count, classes), dtype=np.float32)
    residuals = np.empty((number, group_count), dtype=np.float32)
    group_statistics = np.empty((number, group_count, 4), dtype=np.float32)
    cuda_sync()
    started = time.perf_counter()
    for start in range(0, number, INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, number)
        observed = torch.as_tensor(values[start:stop], device=DEVICE)
        probabilities[start:stop, 0] = classifier(observed).softmax(dim=1).cpu().numpy()
        for index, group in enumerate(groups):
            channels = group['channels']
            masked, available = observed.clone(), torch.ones_like(observed)
            masked[:, channels, :], available[:, channels, :] = 0.0, 0.0
            reconstruction = reconstructor(masked, available)
            repaired = observed.clone()
            repaired[:, channels, :] = reconstruction[:, channels, :]
            probabilities[start:stop, 1 + index] = classifier(masked, available).softmax(dim=1).cpu().numpy()
            probabilities[start:stop, 1 + group_count + index] = classifier(repaired).softmax(dim=1).cpu().numpy()
            residuals[start:stop, index] = (observed[:, channels, :] - reconstruction[:, channels, :]).square().mean(dim=(1, 2)).cpu().numpy()
            signal = values[start:stop, channels, :]
            group_statistics[start:stop, index] = np.stack([
                np.abs(signal).mean(axis=(1, 2)), signal.std(axis=(1, 2)),
                np.abs(np.diff(signal, axis=2)).mean(axis=(1, 2)),
                (signal == 0).mean(axis=(1, 2))], axis=1)
    cuda_sync()
    seconds = time.perf_counter() - started
    assert np.isfinite(probabilities).all() and np.isfinite(residuals).all()
    assert np.allclose(probabilities.sum(axis=2), 1.0, atol=1e-5)
    return probabilities, residuals, group_statistics, seconds


def condition_list():
    conditions = [('clean', None, None)]
    for fault in ['dropout', 'bias', 'gain']:
        for severity in range(3):
            conditions.append((f'{fault}_{severity + 1}', fault, severity))
    return conditions


def create_action_caches(dataset, split, indices, values, y, subjects, window_ids,
                         groups, classes, classifier, reconstructor, run, cache_signature):
    assert split in ['policy', 'validation']
    paths, timing = [], []
    cache_directory = run / 'action_cache'
    cache_directory.mkdir(exist_ok=True)
    action_names = ['keep'] + [f'mask:{g["name"]}' for g in groups] + [f'reconstruct:{g["name"]}' for g in groups]
    for condition, fault, severity in condition_list():
        path = cache_directory / f'{split}__{condition}.npz'
        if path.exists():
            with np.load(path, allow_pickle=False) as saved:
                assert saved['signature'].item() == cache_signature, 'Stale action cache; use a new output version.'
                assert np.array_equal(saved['window_ids'], window_ids[indices])
                seconds = float(saved['full_candidate_inference_seconds'].item())
        else:
            token = zlib.crc32(f'{dataset}|{split}|{condition}'.encode())
            observed = values if fault is None else corrupt_signals(values, groups, np.random.default_rng(SEED + token), fault, severity)
            probabilities, residuals, statistics, seconds = action_outputs(classifier, reconstructor, observed, groups, classes)
            save_npz(path, signature=np.asarray(cache_signature), probabilities=probabilities,
                     reconstruction_residuals=residuals, observed_group_statistics=statistics,
                     y=y[indices], subjects=subjects[indices], window_ids=window_ids[indices],
                     source_indices=indices, condition=np.asarray(condition), split=np.asarray(split),
                     action_names=np.asarray(action_names), full_candidate_inference_seconds=np.asarray(seconds),
                     generation_device=np.asarray(torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'))
        paths.append(path)
        timing.append({'dataset': dataset, 'split': split, 'condition': condition, 'windows': len(indices),
                       'candidate_actions': len(action_names), 'full_candidate_inference_seconds': seconds})
    print(f'  {split}: {len(paths)} action-cache conditions saved or reused.', flush=True)
    return paths, timing


def summarize_validation(dataset, cache_paths, policy_clean_path, classes):
    with np.load(policy_clean_path, allow_pickle=False) as clean:
        thresholds = np.maximum(np.quantile(clean['reconstruction_residuals'], 0.95, axis=0), 1e-8)
    rows = []
    for path in cache_paths:
        with np.load(path, allow_pickle=False) as cached:
            probs = cached['probabilities']
            residuals, y = cached['reconstruction_residuals'], cached['y']
            condition = cached['condition'].item()
        original = probs[:, 0].argmax(axis=1)
        original_correct = original == y
        scaled = residuals / thresholds[None, :]
        suspect = scaled.argmax(axis=1)
        gate = scaled.max(axis=1) > 1.0
        group_count = residuals.shape[1]
        all_predictions = probs.argmax(axis=2)
        correct_actions = all_predictions == y[:, None]
        actions = {
            'keep': np.zeros(len(y), dtype=np.int64),
            'mask_suspect': 1 + suspect,
            'reconstruct_suspect': 1 + group_count + suspect,
            'residual_gate_mask': np.where(gate, 1 + suspect, 0),
            'residual_gate_reconstruct': np.where(gate, 1 + group_count + suspect, 0),
            'confidence_select': probs.max(axis=2).argmax(axis=1),
            'oracle_diagnostic': correct_actions.argmax(axis=1)
        }
        for method, choices in actions.items():
            prediction = all_predictions[np.arange(len(y)), choices]
            correct = prediction == y
            helpful = float(np.mean(~original_correct & correct))
            harmful = float(np.mean(original_correct & ~correct))
            accuracy = float(accuracy_score(y, prediction))
            assert abs((accuracy - original_correct.mean()) - (helpful - harmful)) < 1e-10
            rows.append({'dataset': dataset, 'condition': condition, 'method': method, 'windows': len(y),
                         'accuracy': accuracy, 'macro_f1': macro_f1(y, prediction, classes),
                         'intervention_rate': float(np.mean(choices != 0)),
                         'helpful_rate': helpful, 'harmful_rate': harmful,
                         'harm_given_original_correct': float(np.sum(original_correct & ~correct) / max(1, original_correct.sum()))})
    return rows, thresholds.tolist()


def run_dataset(name):
    print('\n' + name.upper() + ' | fold_0 | seed 2026', flush=True)
    signal_file = cached_input(f'{name}_signals.npz')
    split_file = cached_input(f'{name}_splits.npz')
    normalization_file = cached_input(f'{name}_{FOLD}_normalization.npz')
    with np.load(signal_file, allow_pickle=False) as saved:
        y, subjects, window_ids = saved['y'], saved['subjects'], saved['window_ids']
        raw = saved['X']
    with np.load(split_file, allow_pickle=False) as saved:
        splits = {split: saved[f'{FOLD}_{split}'] for split in ['train', 'policy', 'validation', 'test']}
    with np.load(normalization_file, allow_pickle=False) as saved:
        mean, std = saved['mean'], saved['std']
    # Only development partitions are normalized and passed into model code.
    values = {split: np.ascontiguousarray((raw[indices] - mean[None, :, None]) / std[None, :, None], dtype=np.float32)
              for split, indices in splits.items() if split != 'test'}
    del raw
    groups = protocol['datasets'][name]['sensor_groups']
    classes = len(protocol['datasets'][name]['class_names'])
    channels = values['train'].shape[1]
    for split, indices in splits.items():
        assert len(indices) == len(np.unique(indices))
        for second, other in splits.items():
            if split != second:
                assert not set(subjects[indices].tolist()) & set(subjects[other].tolist())
    run = OUTPUT / f'{name}_{FOLD}_seed_{SEED}'
    run.mkdir(exist_ok=True)
    settings = {'code_version': CODE_VERSION, 'cell_signature': CELL_SIGNATURE, 'dataset': name,
                'fold': FOLD, 'seed': SEED, 'classifier_epochs': CLASSIFIER_EPOCHS,
                'reconstructor_epochs': RECONSTRUCTOR_EPOCHS, 'patience': PATIENCE,
                'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
                'torch_version': str(torch.__version__), 'numpy_version': str(np.__version__),
                'protocol_sha256': completion['protocol_sha256'],
                'input_sha256': {filename: inventory[filename]['sha256'] for filename in
                                 [f'{name}_signals.npz', f'{name}_splits.npz', f'{name}_{FOLD}_normalization.npz']}}
    signature = digest_json(settings)
    configuration_file = run / 'run_configuration.json'
    if configuration_file.exists():
        assert json.loads(configuration_file.read_text())['signature'] == signature, (
            'Training settings or software versions changed. Restore the original settings/environment '
            'to resume, or use a new OUTPUT folder for an independent run.')
    save_json(configuration_file, {'signature': signature, 'settings': settings,
                                   'feature_contract': {'allowed_policy_observables': ['probabilities', 'reconstruction_residuals', 'observed_group_statistics', 'candidate_action_identity'],
                                                        'excluded_policy_inputs': ['y', 'subjects', 'window_ids', 'condition', 'injected_fault_location', 'source_indices']}})
    dataset_seed = SEED + (0 if name == 'dsads' else 100000)
    seed_everything(dataset_seed)
    classifier = ActivityCNN(channels, classes)
    print('  Classifier parameters:', sum(p.numel() for p in classifier.parameters()), flush=True)
    classifier, classifier_history = fit_model('classifier', classifier, values['train'], y[splits['train']],
        values['validation'], y[splits['validation']], groups, classes, run, signature, dataset_seed)
    seed_everything(dataset_seed + 1000)
    reconstructor = SensorReconstructor(channels)
    print('  Reconstructor parameters:', sum(p.numel() for p in reconstructor.parameters()), flush=True)
    reconstructor, reconstructor_history = fit_model('reconstructor', reconstructor, values['train'], y[splits['train']],
        values['validation'], y[splits['validation']], groups, classes, run, signature, dataset_seed + 1000)
    cache_signature = digest_json({'run_signature': signature,
                                  'classifier_best_sha256': sha256_file(run / 'classifier_best.pt'),
                                  'reconstructor_best_sha256': sha256_file(run / 'reconstructor_best.pt')})
    policy_paths, policy_timing = create_action_caches(name, 'policy', splits['policy'], values['policy'],
        y, subjects, window_ids, groups, classes, classifier, reconstructor, run, cache_signature)
    validation_paths, validation_timing = create_action_caches(name, 'validation', splits['validation'], values['validation'],
        y, subjects, window_ids, groups, classes, classifier, reconstructor, run, cache_signature)
    rows, thresholds = summarize_validation(name, validation_paths, run / 'action_cache' / 'policy__clean.npz', classes)
    save_csv(run / 'validation_comparison.csv', pd.DataFrame(rows))
    save_json(run / 'residual_thresholds.json', {'source': '95th percentile of clean policy-partition reconstruction residuals',
                                               'thresholds': thresholds, 'group_names': [g['name'] for g in groups]})
    save_json(run / 'dataset_complete.json', {'signature': signature, 'cache_signature': cache_signature,
              'action_cache_files': len(policy_paths) + len(validation_paths), 'test_predictions_computed': 0,
              'classifier_epochs_completed': len(classifier_history), 'reconstructor_epochs_completed': len(reconstructor_history)})
    del classifier, reconstructor, values
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return rows, policy_timing + validation_timing


def main():
    rows, timings = [], []
    for dataset in ['dsads', 'har']:
        dataset_rows, dataset_timing = run_dataset(dataset)
        rows.extend(dataset_rows)
        timings.extend(dataset_timing)
        save_csv(OUTPUT / 'notebook02_validation_comparison.csv', pd.DataFrame(rows))
        save_csv(OUTPUT / 'notebook02_candidate_inference_cost.csv', pd.DataFrame(timings))
    detailed = pd.DataFrame(rows)
    corrupted = detailed[detailed['condition'] != 'clean']
    aggregate = corrupted.groupby(['dataset', 'method'], as_index=False)[
        ['accuracy', 'macro_f1', 'intervention_rate', 'helpful_rate', 'harmful_rate']].mean()
    clean = detailed[detailed['condition'] == 'clean'][['dataset', 'method', 'accuracy', 'macro_f1']].rename(
        columns={'accuracy': 'clean_accuracy', 'macro_f1': 'clean_macro_f1'})
    aggregate = aggregate.merge(clean, on=['dataset', 'method'])
    save_csv(OUTPUT / 'notebook02_pilot_summary.csv', aggregate)
    pilot = {}
    for name in ['dsads', 'har']:
        part = aggregate[aggregate['dataset'] == name]
        deployable = part[part['method'] != 'oracle_diagnostic']
        best = deployable.loc[deployable['accuracy'].idxmax()]
        oracle = part[part['method'] == 'oracle_diagnostic'].iloc[0]
        keep = part[part['method'] == 'keep'].iloc[0]
        pilot[name] = {'best_validation_baseline_by_accuracy': str(best['method']),
                       'oracle_accuracy_headroom_pp': 100.0 * float(oracle['accuracy'] - best['accuracy']),
                       'oracle_fixable_errors_over_keep_pp': 100.0 * float(oracle['accuracy'] - keep['accuracy'])}
    report = {'code_version': CODE_VERSION, 'fold': FOLD, 'seed': SEED, 'datasets_completed': ['dsads', 'har'],
              'evaluation_split': 'validation', 'conditions_per_dataset': 10,
              'development_faults': ['dropout', 'bias', 'gain'], 'held_out_faults_evaluated': [],
              'test_predictions_computed': 0, 'learned_policy_trained': False,
              'oracle_is_label_assisted_diagnostic': True, 'aggregation': 'equal-weight mean over nine corrupted conditions',
              'candidate_latency_scope': 'classifier and reconstructor for all candidate actions; excludes file IO; not per-baseline latency',
              'pilot': pilot, 'output_directory': str(OUTPUT)}
    save_json(OUTPUT / 'notebook02_run_report.json', report)
    files = []
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and path.name != 'notebook02_artifact_inventory.json' and not path.name.endswith('.partial'):
            files.append({'filename': str(path.relative_to(OUTPUT)), 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
    save_json(OUTPUT / 'notebook02_artifact_inventory.json', files)
    print('\nNOTEBOOK 02 COMPLETE', flush=True)
    print('VALIDATION PILOT: percentages; mean across nine corrupted conditions.')
    display = aggregate[['dataset', 'method', 'clean_macro_f1', 'macro_f1', 'accuracy', 'helpful_rate', 'harmful_rate']].copy()
    display.iloc[:, 2:] = display.iloc[:, 2:] * 100.0
    print(display.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
    for name, value in pilot.items():
        print(f'{name.upper()}: oracle accuracy headroom above best baseline = {value["oracle_accuracy_headroom_pp"]:.2f} pp')
    print('Final test predictions computed: 0 | Held-out stuck-fault evaluation: reserved')
    print('Learned action policy: next notebook | Oracle: diagnostic only')
    print('Outputs:', OUTPUT)
    print('Share the output above and notebook02_pilot_summary.csv, notebook02_run_report.json, notebook02_validation_comparison.csv.')


try:
    main()
except KeyboardInterrupt:
    print('\nNOTEBOOK 02 PAUSED. Completed epochs and conditions are saved. Rerun this same cell to resume.', flush=True)


Mounted at /content/drive
NOTEBOOK 02 | two-dataset validation pilot | one executable cell
Device: Tesla T4 | PyTorch: 2.11.0+cu130
Scope: fold_0, seed 2026; development faults only; final test predictions reserved.

DSADS | fold_0 | seed 2026
  Classifier parameters: 201459
  classifier epoch 01/30 | loss 1.4344 | validation mean Macro-F1 0.7615
  classifier epoch 05/30 | loss 0.1448 | validation mean Macro-F1 0.7413
  classifier epoch 10/30 | loss 0.0948 | validation mean Macro-F1 0.6801
  classifier epoch 12/30 | loss 0.0783 | validation mean Macro-F1 0.7813
  Reconstructor parameters: 93389
  reconstructor epoch 01/25 | loss 0.7920 | validation masked MSE 0.6040
  reconstructor epoch 05/25 | loss 0.4209 | validation masked MSE 0.5028
  reconstructor epoch 10/25 | loss 0.3381 | validation masked MSE 0.4920
  reconstructor epoch 14/25 | loss 0.3278 | validation masked MSE 0.4982
  policy: 10 action-cache conditions saved or reused.
  validation: 10 action-cache conditions saved or re